# 📖 Notebook 2 — Push CDN vs Pull CDN

There are two ways content gets **onto** an edge server:

1. **Pull CDN** — the edge *pulls* content from the origin the first time
   someone asks for it. Classic examples: Cloudflare, Fastly, CloudFront's
   default mode. This is exactly what our nginx `proxy_cache` does.
2. **Push CDN** — you (the content owner) *push* files onto the edge ahead
   of time, before any user asks. Classic examples: Akamai NetStorage,
   Rackspace Cloud Files, most "object storage + CDN" combos like S3 + CloudFront.

> 🧺 Analogy: **pull** is a grocery store that only restocks bread when a
> customer asks for bread. **Push** is a delivery truck that arrives at 5 am
> every day with pre-made orders, whether anyone wants them or not.

## Learning objectives

1. See a pull CDN in action — first request **MISS**, later requests **HIT**.
2. See a push CDN in action — place a file on an edge manually and serve it
   with *zero* origin involvement.
3. Understand when each model makes sense.


## 🛠️ Setup

**Step 1 — Start the lab infrastructure** (from the `01-foundations/cdn/` directory):

```bash
docker compose up -d --build
```

This starts three containers:

| Service | Role                | URL                    |
|---------|---------------------|------------------------|
| origin  | Slow FastAPI server | http://localhost:8000  |
| edge1   | nginx edge POP #1   | http://localhost:8081  |
| edge2   | nginx edge POP #2   | http://localhost:8082  |

**Step 2 — Install Python deps**:

```bash
uv sync
```

**Step 3 — Select the kernel**: in VS Code, click the kernel picker in the
top-right of this notebook and choose the `.venv` interpreter. If it doesn't
show up, reload the window (`Cmd+Shift+P` → *Reload Window*) and try again.


In [ ]:
import time
import statistics
import httpx

ORIGIN = "http://localhost:8000"
EDGE1  = "http://localhost:8081"
EDGE2  = "http://localhost:8082"

def timed_get(url: str, **kwargs) -> tuple[float, httpx.Response]:
    """Return (elapsed_ms, response) for a single GET."""
    t0 = time.perf_counter()
    r = httpx.get(url, timeout=10.0, **kwargs)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return elapsed_ms, r

def show(url: str, elapsed_ms: float, r: httpx.Response) -> None:
    cache = r.headers.get("x-cache-status", "-")
    edge  = r.headers.get("x-edge-server", "-")
    print(f"{url:35s}  {elapsed_ms:7.1f} ms  cache={cache:8s}  edge={edge}")


## 1️⃣ Pull CDN — lazy loading at the edge

`edge1` and `edge2` are both pull-mode caches. When you ask them for an
asset they've never seen, they have to **pull** it from the origin first.
Let's watch that happen on a cache key nobody has warmed yet.

In [ ]:
# A unique query string = a brand-new cache key.
tag = int(time.time())
url = f"{EDGE1}/assets/hello.txt?demo=pull-{tag}"

for i in range(4):
    ms, r = timed_get(url)
    show(f"pull try #{i+1}", ms, r)

You should see something like:

```
pull try #1   ~510 ms  cache=MISS
pull try #2   ~  2 ms  cache=HIT
pull try #3   ~  2 ms  cache=HIT
pull try #4   ~  2 ms  cache=HIT
```

The very first visitor paid the price; everyone after them got the fast
path. This is why pull CDNs are sometimes called **"lazy"** — they only do
work when someone actually asks.

**Advantages of pull**
- Practically zero setup: just point the CDN at your origin.
- The CDN self-manages: new files appear automatically, old ones expire on TTL.
- You pay to move bytes only for content people actually ask for.

**Disadvantages of pull**
- The *first* visitor per edge always pays the full latency — the
  classic "cold cache" problem.
- Your origin must be available when an edge has a miss. An origin outage
  = visible misses everywhere.


## 2️⃣ Push CDN — proactive placement

In a push CDN you upload assets directly to the edge ahead of time, so the
*first* visitor already gets a fast response.

We simulate this by mounting a host folder into each edge container at
`/usr/share/nginx/pushed`, and exposing it as `/pushed/`. Anything we drop
into that folder is served **straight from the edge** — no origin call,
no cache lookup.

Let's push a file onto edge1 only:

In [ ]:
import pathlib, textwrap

# edge1's push drop-zone (mounted as a volume in docker-compose).
push_dir = pathlib.Path("../edge/edge1_pushed")
push_dir.mkdir(parents=True, exist_ok=True)
(push_dir / "hello-push.txt").write_text(
    "This file was PUSHED directly onto edge1 by the content owner.\n"
    "The origin doesn't even have a copy!\n"
)
print("pushed:", list(push_dir.iterdir()))

In [ ]:
# Confirm: the file exists on edge1 but NOT on edge2 or the origin.
for label, url in [
    ("edge1 (pushed) ", f"{EDGE1}/pushed/hello-push.txt"),
    ("edge2 (not pushed)", f"{EDGE2}/pushed/hello-push.txt"),
    ("origin (no such file)", f"{ORIGIN}/assets/hello-push.txt"),
]:
    try:
        ms, r = timed_get(url)
        print(f"{label:22s}  status={r.status_code}  {ms:6.1f} ms  cache={r.headers.get('x-cache-status','-')}")
    except Exception as e:
        print(f"{label:22s}  error: {e}")

Notice:

- edge1 serves the file instantly with `X-Cache-Status: PUSHED` — no origin
  round-trip ever happened.
- edge2 returns 404, because nobody pushed anything there.
- The origin also returns 404 — this asset **only exists at the edge**.

**Advantages of push**
- No cold-miss: the *very first* user gets edge-speed responses.
- Works even if the origin is offline (the edge has the files).
- Great for huge files (videos, installers) you don't want pulled through
  your origin's bandwidth every time a new edge warms up.

**Disadvantages of push**
- You have to run an upload/sync pipeline for every edge.
- You pay storage at every edge whether or not anyone reads the file.
- Invalidation is *your* job — if you forget to push v2, users keep seeing v1.


## 🤔 So which one?

| If you have…                                              | Use…   |
|-----------------------------------------------------------|--------|
| A website with lots of small, frequently-changing assets  | Pull   |
| A handful of big, rarely-changing files (e.g. installers) | Push   |
| A global launch where cold-misses would overload origin   | Push (pre-warm) |
| Unpredictable traffic, minimal ops budget                 | Pull   |

Real systems often **mix both**: pull for general pages and API responses,
push (or pre-warming) for known-hot assets like the homepage hero image
right before a product launch.

➡️ Next: [Notebook 3 — Cache headers & invalidation](./03_cache_headers_and_invalidation.ipynb)